In [ ]:
import pandas as pd
import numpy as np

from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_log_error as RMSE


data_path = "train.csv"
data = pd.read_csv(data_path)
target_thing = np.log1p(data["hours_spent"])

input_data = data.drop(columns=[
    "project_id",
    "hours_spent",
    "log_hours",
    "hours_bucket"
])


qualitative_features = input_data.select_dtypes(include=["object", "string", "category"]).columns.tolist() 
#print(qualitative_features) => ['commit_cadence', 'primary_language', 'framework', 'has_custom_art', 'hosting_type', 'is_mobile_responsive', 'project_category', 'is_multiplayer', 'mentions_ai_api', 'is_clone_or_tutorial', 'author_prior_projects', 'author_account_age_years']

for col in qualitative_features: #fill in missing stuff just in case
    input_data[col] = input_data[col].fillna("Unknown").astype(str)

input_training, input_validation, output_training, output_validation = train_test_split(input_data, target_thing, test_size=0.2, random_state=42)

magicModel = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.03,
    depth=6,
    loss_function="RMSE",
    verbose=100,
    early_stopping_rounds=100
)

magicModel.fit(
    input_training,
    output_training,
    cat_features=qualitative_features,
    eval_set=(input_validation, output_validation)
)

predictionsG=np.expm1(magicModel.predict(input_validation))
predictionsG = np.maximum(predictionsG, 0) #no negatives

true_hours = np.expm1(output_validation)

scoreG=RMSE(true_hours,predictionsG)

print("the score:", scoreG)

results=[]

for category in input_data["project_category"].unique():

    training_check=input_training["project_category"] == category
    validation_check=input_validation["project_category"] == category

    input_category_training = input_data.loc[training_check].drop(
        columns=["project_category"]
    )

    input_category_validation = input_data.loc[validation_check].drop(
            columns=["project_category"]
        )

    output_category_training = output_training.loc[training_check]
    output_category_training = output_validation.loc[validation_check]

    category_columns = [
        c for c in qualitative_features
        if c != "project_category"
    ]

    magicModel = CatBoostRegressor(
        iterations=1000,
        learning_rate=0.03,
        depth=6,
        loss_function="RMSE",
        verbose=100,
        early_stopping_rounds=100
    )

    magicModel.fit(
        input_training,
        output_training,
        cat_features=qualitative_features,
        eval_set=(input_validation, output_validation)
    )

    predictions=np.expm1(magicModel.predict(input_validation))
    predictions = np.maximum(predictions, 0) #no negatives

    true_hours = np.expm1(output_validation)

    score=RMSE(true_hours,predictions)

    qualpredictionG = predictionsG[validation_check.to_numpy()]
    qualscoreG = RMSE(true_hours, qualpredictionG)


0:	learn: 0.9654659	test: 0.8962037	best: 0.8962037 (0)	total: 11.8ms	remaining: 11.8s
100:	learn: 0.6636908	test: 0.6770706	best: 0.6770706 (100)	total: 558ms	remaining: 4.97s
200:	learn: 0.6196252	test: 0.6653937	best: 0.6653937 (200)	total: 1.08s	remaining: 4.28s


KeyboardInterrupt: 